In [ ]:
import medspacy
from medspacy.ner import TargetRule
from medspacy.visualization import visualize_ent

# Load medspacy model
nlp = medspacy.load()
print(nlp.pipe_names)

text = """
"This 4-year-old female presents with a history of infrequent and painful bowel movements over the past several weeks. Parents report stool withholding behavior and passage of hard stools every 4–5 days. No associated vomiting, weight loss, or abdominal distention. On exam, abdomen soft with mild suprapubic fullness, no tenderness. Findings are consistent with functional constipation. No red flags suggesting organic pathology."
"""

# Add rules for target concept extraction
target_matcher = nlp.get_pipe("medspacy_target_matcher")
target_rules = [
    TargetRule("atrial fibrillation", "PROBLEM"),
    TargetRule("atrial fibrillation", "PROBLEM", pattern=[{"LOWER": "afib"}]),
    TargetRule("pneumonia", "PROBLEM"),
    TargetRule("Type II Diabetes Mellitus", "PROBLEM", 
              pattern=[
                  {"LOWER": "type"},
                  {"LOWER": {"IN": ["2", "ii", "two"]}},
                  {"LOWER": {"IN": ["dm", "diabetes"]}},
                  {"LOWER": "mellitus", "OP": "?"}
              ]),
    TargetRule("warfarin", "MEDICATION")
]
target_matcher.add(target_rules)

doc = nlp(text)

visualize_ent(doc)

['medspacy_pyrush', 'medspacy_target_matcher', 'medspacy_context']


In [51]:
from faster_whisper import WhisperModel
import time

audio_file = "Recording.m4a"

model_size = "small"
whisper_model = WhisperModel(model_size, device="cpu", compute_type="int8")

stt_start = time.time()
segments, info = whisper_model.transcribe(audio_file, beam_size=5)
stt_end = time.time()

print(f"Transcription took {stt_end - stt_start:.2f} seconds")
print(type(segments))
for segment in segments:
    print(f"[{segment.start:.2f}s - {segment.end:.2f}s] {segment.text.strip()}")

Transcription took 4.07 seconds
<class 'generator'>
[0.00s - 7.68s] This one year old male is not adequately gaining weight, currently at the third percentile for
[7.68s - 9.18s] weight.
[9.18s - 13.84s] Atenatal and prenatal periods were uneventful.
[13.84s - 17.60s] He was exclusively breastfed until recently.
[17.60s - 23.96s] Clinical observation reveals no signs of organic disease or malabsorption.
[23.96s - 30.36s] This points towards nutritional insufficiency due to inadequate complementary feeding.
[30.36s - 35.44s] Plan involves nutritional counseling and close growth monitoring.


In [13]:
from faster_whisper import WhisperModel
import time
import json
import pandas as pd
from transformers import pipeline
# from ace_tools import display_dataframe_to_user

def extract_clauses_with_advanced_dedup(text, ner_output):
    mapping = {}
    def normalize(c):
        return c.strip().lower().rstrip('.,')
    for ent in ner_output:
        group = ent['entity_group']
        key = group.lower()
        if group == 'SEX':
            mapping['sex'] = ent['word']
            continue
        start, end = ent['start'], ent['end']
        left = max(text.rfind('.', 0, start), text.rfind(',', 0, start))
        rp = text.find('.', end)
        rc = text.find(',', end)
        rights = [pos for pos in (rp, rc) if pos != -1]
        right = min(rights) if rights else len(text)
        clause = text[left+1:right].strip()
        mapping.setdefault(key, []).append(clause)
    for key, clauses in list(mapping.items()):
        if key == 'sex' or not isinstance(clauses, list):
            continue
        unique = []
        for clause in clauses:
            norm = normalize(clause)
            if not norm:
                continue
            if any(norm in normalize(u) for u in unique):
                continue
            unique = [u for u in unique if not normalize(u) in norm]
            unique.append(clause)
        mapping[key] = unique
    return mapping

def group_contiguous_entities(text, ner_output):
    sorted_entities = sorted(ner_output, key=lambda x: x['start'])
    groups = []
    current = None
    for ent in sorted_entities:
        if current is None:
            current = {
                'entity_group': ent['entity_group'],
                'start': ent['start'],
                'end': ent['end'],
                'scores': [ent['score']],
                'words': [ent['word']]
            }
        else:
            if ent['entity_group'] == current['entity_group'] and ent['start'] == current['end']:
                current['end'] = ent['end']
                current['scores'].append(ent['score'])
                current['words'].append(ent['word'])
            else:
                groups.append(current)
                current = {
                    'entity_group': ent['entity_group'],
                    'start': ent['start'],
                    'end': ent['end'],
                    'scores': [ent['score']],
                    'words': [ent['word']]
                }
    if current:
        groups.append(current)
    rows = []
    for g in groups:
        merged_text = text[g['start']:g['end']]
        avg_score = sum(g['scores']) / len(g['scores'])
        rows.append({
            'Entity': g['entity_group'],
            'Text': merged_text,
            'Start': g['start'],
            'End': g['end'],
            'Avg Confidence': round(avg_score, 3)
        })
    return pd.DataFrame(rows)

# Initialize models
model_size = "small"
whisper_model = WhisperModel(model_size, device="cpu", compute_type="int8")
ner_pipeline = pipeline(
    "token-classification",
    model="Clinical-AI-Apollo/Medical-NER",
    aggregation_strategy="simple"
)

audio_file = "Recording.m4a"

# Transcription
t0 = time.time()
segments, _ = whisper_model.transcribe(audio_file, beam_size=5)
t1 = time.time()

# Combine segment texts
full_text = " ".join(segment.text.strip() for segment in segments)

# NER extraction on full transcript
t2 = time.time()
ner_results = ner_pipeline(full_text)
t3 = time.time()

# Extract clauses and group entities
clauses_mapping = extract_clauses_with_advanced_dedup(full_text, ner_results)
df = group_contiguous_entities(full_text, ner_results)

# Display timings and results
print(f"Transcription time: {t1 - t0:.2f}s")
print(f"NER extraction time: {t3 - t2:.2f}s\n")
print("Clauses Mapping:", json.dumps(clauses_mapping, indent=2))
# display_dataframe_to_user("Entity Groups from Full Transcript", df)


Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Transcription time: 3.23s
NER extraction time: 0.20s

Clauses Mapping: {
  "age": [
    "This one year old male is not adequately gaining weight"
  ],
  "sex": "male",
  "diagnostic_procedure": [
    "This one year old male is not adequately gaining weight",
    "currently at the third percentile for weight",
    "Atenatal and prenatal periods were uneventful",
    "Clinical observation reveals no signs of organic disease or malabsorption",
    "Plan involves nutritional counseling and close growth monitoring"
  ],
  "lab_value": [
    "currently at the third percentile for weight",
    "Atenatal and prenatal periods were uneventful"
  ],
  "disease_disorder": [
    "Clinical observation reveals no signs of organic disease or malabsorption",
    "This points towards nutritional insufficiency due to inadequate complementary feeding"
  ],
  "therapeutic_procedure": [
    "Plan involves nutritional counseling and close growth monitoring"
  ],
  "detailed_description": [
    "Plan involves

In [17]:
import re

# Attempt to import word2number for spelled number conversion
try:
    from word2number import w2n
except ImportError:
    w2n = None

# Sample clinical paragraphs with multi-sentence descriptions
sample_paragraphs = [
    ("Dr. Smith noted that the baby was full-term following a normal pregnancy. "
     "At birth, the infant weighed 2.5 kg, cried vigorously, and showed no signs of distress. "
     "The mother had uneventful antenatal history and no perinatal complications were observed."),
    
    ("Upon delivery, the neonate weighed two point five kilograms at birth, with Apgar scores of 8 and 9. "
     "The pregnancy was uncomplicated, and the baby responded well to initial care. "
     "There were no postnatal complications."),
    
    ("This male child was born at 39 weeks gestation. "
     "The newborn weighed two and a half kilos, cried immediately, and was placed on exclusive breastfeeding. "
     "The family history revealed no consanguinity."),
    
    ("The infant was delivered via NVD at term. "
     "He weighed 2500 grams at birth and exhibited good muscle tone. "
     "No neonatal resuscitation was required, and he was breastfed up to six months."),
    
    ("Dr. Lee documented that the baby was born weighing three kilos after a spontaneous vaginal delivery. "
     "The antenatal course was uneventful, and the child cried at birth. "
     "No birth complications were noted.")
]

def extract_birth_weight(text: str) -> float:
    """
    Extracts birth weight in kilograms from a clinical text.
    Returns the weight in kg, or None if no weight is found.
    """
    # Numeric pattern for kg or g
    numeric_match = re.search(r'(\d+(?:[\d,]*)(?:\.\d+)?)\s*(kg|kilograms|g|grams|kilos)', text, re.IGNORECASE)
    if numeric_match:
        value = float(numeric_match.group(1).replace(',', ''))
        unit = numeric_match.group(2).lower()
        if unit in ['g', 'grams']:
            value = value / 1000
        return value

    # Spelled-out number pattern for kg/kilos
    if w2n:
        spelled_match = re.search(r'([a-z\s\-\d]+?)\s*(kg|kilograms|kilos)', text, re.IGNORECASE)
        if spelled_match:
            words = spelled_match.group(1)
            try:
                num = w2n.word_to_num(words)
                return float(num)
            except ValueError:
                pass

    return None

# Experiment the function on the multi-sentence paragraphs
results = []
for para in sample_paragraphs:
    weight_kg = extract_birth_weight(para)
    results.append({"text": para, "birth_weight_kg": weight_kg})

import pandas as pd
# from ace_tools import display_dataframe_to_user

df = pd.DataFrame(results)
df
# display_dataframe_to_user("Birth Weight Extraction Results", df)


,text,birth_weight_kg
0,Dr. Smith noted that the baby was full-term fo...,2.5
1,"Upon delivery, the neonate weighed two point f...",2.5
2,This male child was born at 39 weeks gestation...,2.0
3,The infant was delivered via NVD at term. He w...,2.5
4,Dr. Lee documented that the baby was born weig...,3.0


In [62]:
# Package: clinical_extractor
# Directory structure:
# clinical_extractor/
# ├── __init__.py
# ├── config.py
# ├── pure_spacy_extractor.py
# ├── medspacy_extractor.py
# ├── test_extractor.py
# └── test_benchmark.py

# File: clinical_extractor/config.py
# Shared keyword mappings for radio fields
KEYWORD_MAP = {
    "conception_mode": {
        "natural": "Natural",
        "naturally": "Natural",
        "spontaneous": "Natural",
        "assisted": "Assisted",
        "ivf": "Assisted",
        "in vitro fertilization": "Assisted",
        "art": "Assisted",
        "assisted reproductive technology": "Assisted",
    },
    "delivery_mode": {
        "nvd": "NVD",
        "normal vaginal": "NVD",
        "spontaneous vaginal": "NVD",
        "vaginal delivery": "NVD",
        "vaginal": "NVD",
        "cesarean": "LSCS",
        "caesarean": "LSCS",
        "c section": "LSCS",
        "c-section": "LSCS",
        "csection": "LSCS",
        "elective c-section": "LSCS",
        "surgical delivery": "LSCS",
        "abdominal delivery": "LSCS",
        "assisted": "Assisted",
    },
    "term": {
        "term": "Term",
        "full-term": "Term",
        "full term": "Term",
        "at term": "Term",
        "preterm": "Preterm",
        "premature": "Preterm",
        "premature birth": "Preterm",
        "preemie": "Preterm",
        "premo": "Preterm",
    },
    "cried_at_birth": {
        "cried": "Yes",
        "cried immediately": "Yes",
        "cried vigorously": "Yes",
        "did not cry": "No",
        "didn't cry": "No",
        "failed to cry": "No",
        "no cry": "No",
        "apgar 0": "No",
    },
}

# File: clinical_extractor/pure_spacy_extractor.py
import spacy
from spacy.matcher import PhraseMatcher
# from .config import KEYWORD_MAP

# Load a lightweight English model
nlp = spacy.load("en_core_web_sm")

# Build PhraseMatcher patterns for each field
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
for field, synonyms in KEYWORD_MAP.items():
    patterns = [nlp.make_doc(expr) for expr in synonyms.keys()]
    matcher.add(field, patterns)


def extract_radio_fields(text: str) -> dict:
    """
    Pure spaCy extractor for radio-button fields via PhraseMatcher.
    Returns a dict with field:value mappings (first hit per field).
    """
    doc = nlp(text)
    results = {}
    for match_id, start, end in matcher(doc):
        field = nlp.vocab.strings[match_id]
        span_text = doc[start:end].text.lower()
        if field not in results:
            results[field] = KEYWORD_MAP[field].get(span_text)
    return results

# File: clinical_extractor/medspacy_extractor.py
import re
import spacy
from spacy.matcher import PhraseMatcher
# from .config import KEYWORD_MAP

# Initialize spaCy model with sentence segmentation
doc_nlp = spacy.load("en_core_web_sm")
doc_nlp.add_pipe("sentencizer", first=True)

# Build PhraseMatcher patterns for each field
detector = PhraseMatcher(doc_nlp.vocab, attr="LOWER")
for field, synonyms in KEYWORD_MAP.items():
    patterns = [doc_nlp.make_doc(expr) for expr in synonyms.keys()]
    detector.add(field, patterns)

# Context keywords for disambiguation
CONTEXT_KEYWORDS = {
    "conception_mode": ["conceived", "fertilization", "art"],
    "delivery_mode": ["deliver", "delivery", "section", "vaginal"],
}


def extract_radio_fields_medspacy(text: str) -> dict:
    """
    Extract radio fields with negation and context filtering.
    Returns a dict with field:value mappings.
    """
    results = {}
    doc = doc_nlp(text)
    for sent in doc.sents:
        sent_text = sent.text.lower()
        matches = detector(sent)
        for match_id, start, end in matches:
            field = doc_nlp.vocab.strings[match_id]
            span = sent[start:end]
            span_text = span.text.lower()
            # Handle negation for 'cried_at_birth'
            if field == "cried_at_birth":
                neg = bool(re.search(r"\b(did not cry|didn't cry|failed to cry|no cry)\b", sent_text))
                results[field] = "No" if neg else "Yes"
                continue
            # Map only if in appropriate context for ambiguous terms
            option = KEYWORD_MAP[field].get(span_text)
            if option:
                if span_text == "assisted":
                    if not any(ctx in sent_text for ctx in CONTEXT_KEYWORDS.get(field, [])):
                        continue
                if field not in results:
                    results[field] = option
    return results

# File: clinical_extractor/test_extractor.py
from radio_extractor.spacy_extractor import extract_radio_fields as pure_extract
from radio_extractor.medspacy_extractor import extract_radio_fields_medspacy as med_extract

# Sample clinical paragraphs
demo_texts = [
    "The baby was conceived naturally and delivered at term via NVD. He cried at birth.",
    "This infant was conceived via IVF and delivered by caesarean section. The newborn did not cry.",
    "A preterm male born by spontaneous vaginal delivery. Crying vigorously at birth.",
    "Premature birth at 35 weeks, no cry was noted. Delivery: assisted.",
    "Infant conceived by ART, delivered normal vaginally at term, cried immediately."
]

if __name__ == "__main__":
    print("Comparison of pure spaCy vs medSpaCy-like extractors:\n")
    for text in demo_texts:
        print(f"Text: {text}")
        print(f"Pure spaCy:     {pure_extract(text)}")
        print(f"medSpaCy-like: {med_extract(text)}\n")

# File: clinical_extractor/test_benchmark.py
import time
from radio_extractor.spacy_extractor import extract_radio_fields as pure_extract
from radio_extractor.medspacy_extractor import extract_radio_fields_medspacy as med_extract

# Ground truth annotations for benchmarking
benchmark_cases = [
    {
        "text": "The baby was conceived naturally and delivered at term via NVD. He cried at birth.",
        "expected": {"conception_mode": "Natural", "delivery_mode": "NVD", "term": "Term", "cried_at_birth": "Yes"}
    },
    {
        "text": "This infant was conceived via IVF and delivered by caesarean section. The newborn did not cry.",
        "expected": {"conception_mode": "Assisted", "delivery_mode": "LSCS", "term": None, "cried_at_birth": "No"}
    },
    {
        "text": "A preterm male born by spontaneous vaginal delivery. Crying vigorously at birth.",
        "expected": {"conception_mode": None, "delivery_mode": "NVD", "term": "Preterm", "cried_at_birth": "Yes"}
    },
    {
        "text": "Premature birth at 35 weeks, no cry was noted. Delivery: assisted.",
        "expected": {"conception_mode": None, "delivery_mode": None, "term": "Preterm", "cried_at_birth": "No"}
    },
    {
        "text": "Infant conceived by ART, delivered normal vaginally at term, cried immediately.",
        "expected": {"conception_mode": "Assisted", "delivery_mode": "NVD", "term": "Term", "cried_at_birth": "Yes"}
    }
]

# Benchmark function
def evaluate(extractor, cases):
    correct = 0
    total = 0
    start = time.perf_counter()
    for case in cases:
        res = extractor(case["text"])
        for field, expected in case["expected"].items():
            total += 1
            if res.get(field) == expected:
                correct += 1
    elapsed = time.perf_counter() - start
    accuracy = correct / total * 100
    return accuracy, elapsed

if __name__ == "__main__":
    print("Benchmarking extraction accuracy and speed:\n")
    pure_acc, pure_time = evaluate(pure_extract, benchmark_cases)
    med_acc, med_time = evaluate(med_extract, benchmark_cases)

    print(f"Pure spaCy extractor -> Accuracy: {pure_acc:.2f}% | Time: {pure_time:.4f}s")
    print(f"medSpaCy-like extractor -> Accuracy: {med_acc:.2f}% | Time: {med_time:.4f}s")
    print("\nOverall: ")
    if pure_acc > med_acc:
        print("Pure spaCy is more accurate.")
    elif med_acc > pure_acc:
        print("medSpaCy-like is more accurate.")
    else:
        print("Both have equal accuracy.")
    if pure_time < med_time:
        print("Pure spaCy is faster.")
    else:
        print("medSpaCy-like is faster.")


spaCy model loaded.
Sentencizer added to spaCy pipeline.
Patterns added to PhraseMatcher.
Context keywords defined.
Comparison of pure spaCy vs medSpaCy-like extractors:

Text: The baby was conceived naturally and delivered at term via NVD. He cried at birth.
Pure spaCy:     {'term': 'Term', 'delivery_mode': 'NVD', 'cried_at_birth': 'Yes'}
Processing sentence: the baby was conceived naturally and delivered at term via nvd.
Matches found: [(622592083565054148, 4, 5), (4519742297340331040, 6, 9), (4519742297340331040, 7, 9), (4519742297340331040, 8, 9), (9586643516838484807, 10, 11)]
Match ID: 622592083565054148, Start: 4, End: 5
Field: conception_mode
Span: naturally
Span text: naturally
Option found: Natural
Mapping naturally to conception_mode: Natural
Adding conception_mode to results.
Match ID: 4519742297340331040, Start: 6, End: 9
Field: term
Span: delivered at term
Span text: delivered at term
Option found: Term
Mapping delivered at term to term: Term
Adding term to results.
Match

In [27]:
import re
import pandas as pd
# from ace_tools import display_dataframe_to_user

# Updated extraction function with improved patterns and normalization
def extract_text_fields(text: str) -> dict:
    results = {}
    lower_text = text.lower()

    # Pedigree: handle "pedigree: no ..." and "family history is negative for ..."
    m = re.search(r'(?:pedigree.*? no|family history is negative for)\s*([\w\s-]+?)(?:;|\.|\band\b)', text, re.IGNORECASE)
    if m:
        clause = m.group(1).strip().lower()
        if 'negative for' in m.group(0).lower():
            results['pedigree'] = f"negative for {clause}"
        else:
            results['pedigree'] = f"no {clause}"

    # Consanguinity
    m = re.search(r'\b(no history of consanguinity|parents are unrelated|no consanguinity reported)\b', text, re.IGNORECASE)
    if m:
        results['consanguinity'] = m.group(1).strip().lower()

    # Antenatal History
    m = re.search(r'antenatal (?:history|period)(?: was|:)?(?: remarkable only for\s*)?(.*?)(?:\.|$)', text, re.IGNORECASE)
    if m:
        results['antenatal_history'] = m.group(1).strip().rstrip('.').lower()

    # Perinatal History
    m = re.search(r'perinatal (?:events )?(?:history:)?\s*(.*?)(?:\.|$)', text, re.IGNORECASE)
    if m:
        results['perinatal_history'] = m.group(1).strip().rstrip('.').lower()

    # Postnatal Complications
    m = re.search(r'\bno postnatal complications\b', text, re.IGNORECASE)
    if m:
        results['postnatal_complications'] = m.group(0).lower()
    else:
        m = re.search(r'postnatal complications (?:occurred|included)\s*(.*?)(?:\.|$)', text, re.IGNORECASE)
        if m:
            results['postnatal_complications'] = m.group(1).strip().lower()

    # Breastfed Upto
    m = re.search(r'breastfed (?:the infant )?(?:up to|for)\s*([\w\s]+?)(?:\.|$)', text, re.IGNORECASE)
    if m:
        results['breastfed_upto'] = m.group(1).strip().lower()

    return results

# Sample test cases
TEST_CASES = [
    {
        "text": (
            "Dr. Rao documented a clean pedigree with no familial genetic disorders and no history of consanguinity. "
            "Antenatal history was remarkable only for a mild urinary tract infection treated at 32 weeks. "
            "Perinatal events were unremarkable. There were no postnatal complications, and the mother breastfed the infant up to eight weeks."
        ),
        "expected": {
            "pedigree": "no familial genetic disorders",
            "consanguinity": "no history of consanguinity",
            "antenatal_history": "mild urinary tract infection treated at 32 weeks",
            "perinatal_history": "unremarkable",
            "postnatal_complications": "no postnatal complications",
            "breastfed_upto": "eight weeks"
        }
    },
    {
        "text": (
            "Family history is negative for neuromuscular diseases; parents are unrelated. "
            "Antenatal period included gestational diabetes managed with diet. "
            "Perinatal history: meconium-stained liquor but no distress. "
            "No postnatal complications occurred. "
            "The baby was breastfed for four months."
        ),
        "expected": {
            "pedigree": "negative for neuromuscular diseases",
            "consanguinity": "parents are unrelated",
            "antenatal_history": "gestational diabetes managed with diet",
            "perinatal_history": "meconium-stained liquor but no distress",
            "postnatal_complications": "no postnatal complications",
            "breastfed_upto": "four months"
        }
    },
    {
        "text": (
            "Pedigree: no known hereditary disorders; no consanguinity reported. "
            "During the antenatal period she had preeclampsia in the third trimester. "
            "Perinatal history: mild shoulder dystocia requiring gentle traction. "
            "Postnatal complications included transient tachypnea of the newborn. "
            "The infant was breastfed up to six months."
        ),
        "expected": {
            "pedigree": "no known hereditary disorders",
            "consanguinity": "no consanguinity reported",
            "antenatal_history": "preeclampsia in the third trimester",
            "perinatal_history": "mild shoulder dystocia requiring gentle traction",
            "postnatal_complications": "transient tachypnea of the newborn",
            "breastfed_upto": "six months"
        }
    }
]

# Evaluate extraction
rows = []
for case in TEST_CASES:
    extracted = extract_text_fields(case['text'])
    row = {'text': case['text']}
    for field, expected in case['expected'].items():
        row[f'extracted_{field}'] = extracted.get(field)
        row[f'expected_{field}'] = expected
        row[f'{field}_match'] = extracted.get(field) == expected
    rows.append(row)

df = pd.DataFrame(rows)
# display_dataframe_to_user("Updated Birth History Text Extraction Results", df)
df


,text,extracted_pedigree,expected_pedigree,pedigree_match,extracted_consanguinity,expected_consanguinity,consanguinity_match,extracted_antenatal_history,expected_antenatal_history,antenatal_history_match,extracted_perinatal_history,expected_perinatal_history,perinatal_history_match,extracted_postnatal_complications,expected_postnatal_complications,postnatal_complications_match,extracted_breastfed_upto,expected_breastfed_upto,breastfed_upto_match
0,Dr. Rao documented a clean pedigree with no fa...,no familial genetic disorders,no familial genetic disorders,True,no history of consanguinity,no history of consanguinity,True,a mild urinary tract infection treated at 32 w...,mild urinary tract infection treated at 32 weeks,False,were unremarkable,unremarkable,False,no postnatal complications,no postnatal complications,True,eight weeks,eight weeks,True
1,Family history is negative for neuromuscular d...,negative for neuromuscular diseases,negative for neuromuscular diseases,True,parents are unrelated,parents are unrelated,True,included gestational diabetes managed with diet,gestational diabetes managed with diet,False,meconium-stained liquor but no distress,meconium-stained liquor but no distress,True,no postnatal complications,no postnatal complications,True,four months,four months,True
2,Pedigree: no known hereditary disorders; no co...,no known hereditary disorders,no known hereditary disorders,True,no consanguinity reported,no consanguinity reported,True,she had preeclampsia in the third trimester,preeclampsia in the third trimester,False,mild shoulder dystocia requiring gentle traction,mild shoulder dystocia requiring gentle traction,True,transient tachypnea of the newborn,transient tachypnea of the newborn,True,six months,six months,True


In [28]:
import re
import pandas as pd
# from ace_tools import display_dataframe_to_user

# Updated extraction function from medSpaCy-like MVP
def extract_text_fields(text: str) -> dict:
    results = {}
    # Pedigree
    m = re.search(r'(?:pedigree.*? no|family history is negative for)\s*([\w\s-]+?)(?:;|\.|\band\b)', text, re.IGNORECASE)
    if m:
        clause = m.group(1).strip().lower()
        if 'negative for' in m.group(0).lower():
            results['pedigree'] = f"negative for {clause}"
        else:
            results['pedigree'] = f"no {clause}"

    # Consanguinity
    m = re.search(r'\b(no history of consanguinity|parents are unrelated|no consanguinity reported)\b', text, re.IGNORECASE)
    if m:
        results['consanguinity'] = m.group(1).strip().lower()

    # Antenatal History
    m = re.search(r'antenatal (?:history|period)(?: was|:)?(?: remarkable only for\s*)?(.*?)(?:\.|$)', text, re.IGNORECASE)
    if m:
        results['antenatal_history'] = m.group(1).strip().rstrip('.').lower()

    # Perinatal History
    m = re.search(r'perinatal (?:events )?(?:history:)?\s*(.*?)(?:\.|$)', text, re.IGNORECASE)
    if m:
        results['perinatal_history'] = m.group(1).strip().rstrip('.').lower()

    # Postnatal Complications
    m = re.search(r'\bno postnatal complications\b', text, re.IGNORECASE)
    if m:
        results['postnatal_complications'] = m.group(0).lower()
    else:
        m = re.search(r'postnatal complications (?:occurred|included)\s*(.*?)(?:\.|$)', text, re.IGNORECASE)
        if m:
            results['postnatal_complications'] = m.group(1).strip().lower()

    # Breastfed Upto
    m = re.search(r'breastfed (?:the infant )?(?:up to|for)\s*([\w\s]+?)(?:\.|$)', text, re.IGNORECASE)
    if m:
        results['breastfed_upto'] = m.group(1).strip().lower()

    return results

# Original and adversarial test cases
TEST_CASES = [
    # Original cases...
    {
        "text": (
            "Dr. Rao documented a clean pedigree with no familial genetic disorders and no history of consanguinity. "
            "Antenatal history was remarkable only for a mild urinary tract infection treated at 32 weeks. "
            "Perinatal events were unremarkable. There were no postnatal complications, and the mother breastfed the infant up to eight weeks."
        ),
        "expected": {
            "pedigree": "no familial genetic disorders",
            "consanguinity": "no history of consanguinity",
            "antenatal_history": "mild urinary tract infection treated at 32 weeks",
            "perinatal_history": "unremarkable",
            "postnatal_complications": "no postnatal complications",
            "breastfed_upto": "eight weeks"
        }
    },
    {
        "text": (
            "Family history is negative for neuromuscular diseases; parents are unrelated. "
            "Antenatal period included gestational diabetes managed with diet. "
            "Perinatal history: meconium-stained liquor but no distress. "
            "No postnatal complications occurred. "
            "The baby was breastfed for four months."
        ),
        "expected": {
            "pedigree": "negative for neuromuscular diseases",
            "consanguinity": "parents are unrelated",
            "antenatal_history": "gestational diabetes managed with diet",
            "perinatal_history": "meconium-stained liquor but no distress",
            "postnatal_complications": "no postnatal complications",
            "breastfed_upto": "four months"
        }
    },
    {
        "text": (
            "Pedigree: no known hereditary disorders; no consanguinity reported. "
            "During the antenatal period she had preeclampsia in the third trimester. "
            "Perinatal history: mild shoulder dystocia requiring gentle traction. "
            "Postnatal complications included transient tachypnea of the newborn. "
            "The infant was breastfed up to six months."
        ),
        "expected": {
            "pedigree": "no known hereditary disorders",
            "consanguinity": "no consanguinity reported",
            "antenatal_history": "preeclampsia in the third trimester",
            "perinatal_history": "mild shoulder dystocia requiring gentle traction",
            "postnatal_complications": "transient tachypnea of the newborn",
            "breastfed_upto": "six months"
        }
    },
    # Adversarial cases...
    {
        "text": (
            "No issues in family lineage; unrelated parents. Antenatal included severe vomiting and dehydration at week 28. "
            "Perinatal: cord prolapse managed successfully. Postnatal: baby needed phototherapy for jaundice. "
            "Breastfed exclusively until the end of Month Seven."
        ),
        "expected": {
            "pedigree": None,  # regex won't catch 'No issues in family lineage'
            "consanguinity": "unrelated parents",
            "antenatal_history": "severe vomiting and dehydration at week 28",
            "perinatal_history": "cord prolapse managed successfully",
            "postnatal_complications": "baby needed phototherapy for jaundice",
            "breastfed_upto": "month seven"
        }
    },
    {
        "text": (
            "Lineage: no genetic red flags. Antenatal history: hypertension detected T1; perinatal history unremarkable. "
            "After birth, no complications seen; breastfed for 2 mos."
        ),
        "expected": {
            "pedigree": None,  # missing 'pedigree' keyword
            "consanguinity": None,  # not mentioned
            "antenatal_history": "hypertension detected t1",
            "perinatal_history": "unremarkable",
            "postnatal_complications": "no complications seen",
            "breastfed_upto": "2 mos"
        }
    }
]

# Run extraction tests
results = []
total, correct = 0, 0
for case in TEST_CASES:
    extracted = extract_text_fields(case["text"])
    for field, expected in case["expected"].items():
        total += 1
        actual = extracted.get(field)
        match = (actual == expected)
        if match:
            correct += 1
    row = {"text": case["text"], **extracted}
    results.append(row)

accuracy = correct / total * 100

# Display detailed table
df = pd.DataFrame(results)
# display_dataframe_to_user("Robust Birth History Extraction", df)
df

# Print summary
print(f"Accuracy: {accuracy:.2f}% over {len(TEST_CASES)} test cases and {total} fields")


Accuracy: 60.00% over 5 test cases and 30 fields


In [41]:
# File: test_birth_history_examples.py
# Test suite to validate Birth History text extraction accuracy using individual functions

import re
import json

# Sample paragraphs and expected values for Birth History fields
TEST_CASES = [
    {
        "text": (
            "Dr. Rao documented a clean pedigree with no familial genetic disorders and no history of consanguinity. "
            "Antenatal history was remarkable only for a mild urinary tract infection treated at 32 weeks. "
            "Perinatal events were unremarkable. There were no postnatal complications, and the mother breastfed the infant up to eight weeks."
        ),
        "expected": {
            "pedigree": "no familial genetic disorders",
            "consanguinity": "no history of consanguinity",
            "antenatal_history": "mild urinary tract infection treated at 32 weeks",
            "perinatal_history": "unremarkable",
            "postnatal_complications": "no postnatal complications",
            "breastfed_upto": "eight weeks"
        }
    },
    {
        "text": (
            "Family history is negative for neuromuscular diseases; parents are unrelated. "
            "Antenatal period included gestational diabetes managed with diet. "
            "Perinatal history: meconium-stained liquor but no distress. "
            "No postnatal complications occurred. "
            "The baby was breastfed for four months."
        ),
        "expected": {
            "pedigree": "negative for neuromuscular diseases",
            "consanguinity": "parents are unrelated",
            "antenatal_history": "gestational diabetes managed with diet",
            "perinatal_history": "meconium-stained liquor but no distress",
            "postnatal_complications": "no postnatal complications",
            "breastfed_upto": "four months"
        }
    },
    {
        "text": (
            "Pedigree: no known hereditary disorders; no consanguinity reported. "
            "During the antenatal period she had preeclampsia in the third trimester. "
            "Perinatal history: mild shoulder dystocia requiring gentle traction. "
            "Postnatal complications included transient tachypnea of the newborn. "
            "The infant was breastfed up to six months."
        ),
        "expected": {
            "pedigree": "no known hereditary disorders",
            "consanguinity": "no consanguinity reported",
            "antenatal_history": "preeclampsia in the third trimester",
            "perinatal_history": "mild shoulder dystocia requiring gentle traction",
            "postnatal_complications": "transient tachypnea of the newborn",
            "breastfed_upto": "six months"
        }
    },
    {
        # Adversarial Scenario 1
        "text": (
            "No issues in family lineage; unrelated parents. Antenatal included severe vomiting and dehydration at week 28. "
            "Perinatal: cord prolapse managed successfully. Postnatal: baby needed phototherapy for jaundice. "
            "Breastfed exclusively until the end of Month Seven."
        ),
        "expected": {
            "pedigree": None,
            "consanguinity": "unrelated parents",
            "antenatal_history": "severe vomiting and dehydration at week 28",
            "perinatal_history": "cord prolapse managed successfully",
            "postnatal_complications": "baby needed phototherapy for jaundice",
            "breastfed_upto": "month seven"
        }
    },
    {
        # Adversarial Scenario 2
        "text": (
            "Lineage: no genetic red flags. Antenatal history: hypertension detected t1; perinatal history unremarkable. "
            "After birth, no complications seen; breastfed for 2 mos."
        ),
        "expected": {
            "pedigree": None,
            "consanguinity": None,
            "antenatal_history": "hypertension detected t1",
            "perinatal_history": "unremarkable",
            "postnatal_complications": "no complications seen",
            "breastfed_upto": "2 mos"
        }
    },
    {
        # Normal Scenario 3
        "text": (
            "Genetic lineage appears unremarkable; no consanguineous relation. Antenatal history included hyperemesis gravidarum in early pregnancy. "
            "Perinatal history documented preterm rupture of membranes at 34 weeks. Postnatal complications: neonatal sepsis treated with intravenous antibiotics. "
            "The infant was breastfed for ten weeks."
        ),
        "expected": {
            "pedigree": None,
            "consanguinity": "no consanguineous relation",
            "antenatal_history": "hyperemesis gravidarum in early pregnancy",
            "perinatal_history": "preterm rupture of membranes at 34 weeks",
            "postnatal_complications": "neonatal sepsis treated with intravenous antibiotics",
            "breastfed_upto": "ten weeks"
        }
    },
]

# Extraction functions for each field

def extract_pedigree(text: str) -> str:
    m = re.search(r'(?:pedigree[:\s]*no|family history is negative for)\s*([\w\s-]+?)(?:;|\.|$)', text, re.IGNORECASE)
    if m:
        clause = m.group(1).strip().lower()
        return clause if 'negative for' in m.group(0).lower() else f'no {clause}'
    return None


def extract_consanguinity(text: str) -> str:
    m = re.search(r'\b(no history of consanguinity|parents are unrelated|no consanguinity reported)\b', text, re.IGNORECASE)
    return m.group(1).lower() if m else None


def extract_antenatal_history(text: str) -> str:
    m = re.search(r'antenatal (?:history|period)(?: was|:)?\s*(.*?)(?:\.|$)', text, re.IGNORECASE)
    return m.group(1).strip().lower() if m else None


def extract_perinatal_history(text: str) -> str:
    m = re.search(r'perinatal (?:history:|events )?\s*(.*?)(?:\.|$)', text, re.IGNORECASE)
    return m.group(1).strip().lower() if m else None


def extract_postnatal_complications(text: str) -> str:
    m = re.search(r'\bno postnatal complications\b', text, re.IGNORECASE)
    if m:
        return m.group(0).lower()
    m = re.search(r'postnatal (?:complications[:]?|)\s*(?:occurred\s*)?(.*?)(?:\.|$)', text, re.IGNORECASE)
    return m.group(1).strip().lower() if m else None


def extract_breastfed_upto(text: str) -> str:
    m = re.search(r'breastfed (?:the infant )?(?:up to|for)\s*([\w\s]+?)(?:\.|$)', text, re.IGNORECASE)
    return m.group(1).strip().lower() if m else None


def extract_all_fields(text: str) -> dict:
    return {
        'pedigree': extract_pedigree(text),
        'consanguinity': extract_consanguinity(text),
        'antenatal_history': extract_antenatal_history(text),
        'perinatal_history': extract_perinatal_history(text),
        'postnatal_complications': extract_postnatal_complications(text),
        'breastfed_upto': extract_breastfed_upto(text),
    }

# Run tests and report accuracy with flexible substring matching
if __name__ == '__main__':
    total_fields = 0
    correct_fields = 0
    for idx, case in enumerate(TEST_CASES, 1):
        extracted = extract_all_fields(case['text'])
        print(f"\n--- Test Case {idx} ---")
        for field, expected in case['expected'].items():
            actual = extracted.get(field)
            # Flexible matching: allow either string to contain the other when both are present
            if expected is None:
                match = actual is None
            elif actual is None:
                match = False
            else:
                match = isinstance(actual, str) and (
                    expected in actual or actual in expected
                )
            status = '✓' if match else '✗'
            print(f"{field:25}: expected={expected!r}, actual={actual!r} {status}")
            total_fields += 1
            if match:
                correct_fields += 1
    accuracy = correct_fields / total_fields * 100 if total_fields > 0 else 0
    print(f"\nOverall accuracy: {accuracy:.2f}% ({correct_fields}/{total_fields})")


--- Test Case 1 ---
pedigree                 : expected='no familial genetic disorders', actual=None ✗
consanguinity            : expected='no history of consanguinity', actual='no history of consanguinity' ✓
antenatal_history        : expected='mild urinary tract infection treated at 32 weeks', actual='remarkable only for a mild urinary tract infection treated at 32 weeks' ✓
perinatal_history        : expected='unremarkable', actual='were unremarkable' ✓
postnatal_complications  : expected='no postnatal complications', actual='no postnatal complications' ✓
breastfed_upto           : expected='eight weeks', actual='eight weeks' ✓

--- Test Case 2 ---
pedigree                 : expected='negative for neuromuscular diseases', actual='neuromuscular diseases' ✓
consanguinity            : expected='parents are unrelated', actual='parents are unrelated' ✓
antenatal_history        : expected='gestational diabetes managed with diet', actual='included gestational diabetes managed with diet' ✓


Regular expression

In [43]:
# File: test_birth_history_examples.py
# Test suite to validate Birth History text extraction accuracy with regex + medSpaCy fallback

import re
from test_birth_history_examples import TEST_CASES
# Attempt to import spaCy and medSpaCy; if unavailable, medSpaCy fallback is disabled
try:
    import spacy
    import medspacy  # registers medSpaCy components
    from spacy.pipeline import EntityRuler
    SPACY_AVAILABLE = True
except ImportError:
    SPACY_AVAILABLE = False

# In-file test cases
# TEST_CASES = [
#     {
#         "text": (
#             "Dr. Rao documented a clean pedigree with no familial genetic disorders and no history of consanguinity. "
#             "Antenatal history was remarkable only for a mild urinary tract infection treated at 32 weeks. "
#             "Perinatal events were unremarkable. There were no postnatal complications, and the mother breastfed the infant up to eight weeks."
#         ),
#         "expected": {
#             "pedigree": "no familial genetic disorders",
#             "consanguinity": "no history of consanguinity",
#             "antenatal_history": "mild urinary tract infection treated at 32 weeks",
#             "perinatal_history": "unremarkable",
#             "postnatal_complications": "no postnatal complications",
#             "breastfed_upto": "eight weeks"
#         }
#     },
#     {
#         "text": (
#             "Family history is negative for neuromuscular diseases; parents are unrelated. "
#             "Antenatal period included gestational diabetes managed with diet. "
#             "Perinatal history: meconium-stained liquor but no distress. "
#             "No postnatal complications occurred. "
#             "The baby was breastfed for four months."
#         ),
#         "expected": {
#             "pedigree": "negative for neuromuscular diseases",
#             "consanguinity": "parents are unrelated",
#             "antenatal_history": "gestational diabetes managed with diet",
#             "perinatal_history": "meconium-stained liquor but no distress",
#             "postnatal_complications": "no postnatal complications",
#             "breastfed_upto": "four months"
#         }
#     },
#     {
#         "text": (
#             "Pedigree: no known hereditary disorders; no consanguinity reported. "
#             "During the antenatal period she had preeclampsia in the third trimester. "
#             "Perinatal history: mild shoulder dystocia requiring gentle traction. "
#             "Postnatal complications included transient tachypnea of the newborn. "
#             "The infant was breastfed up to six months."
#         ),
#         "expected": {
#             "pedigree": "no known hereditary disorders",
#             "consanguinity": "no consanguinity reported",
#             "antenatal_history": "preeclampsia in the third trimester",
#             "perinatal_history": "mild shoulder dystocia requiring gentle traction",
#             "postnatal_complications": "transient tachypnea of the newborn",
#             "breastfed_upto": "six months"
#         }
#     },
#     {
#         # Adversarial Scenario 1
#         "text": (
#             "No issues in family lineage; unrelated parents. Antenatal included severe vomiting and dehydration at week 28. "
#             "Perinatal: cord prolapse managed successfully. Postnatal: baby needed phototherapy for jaundice. "
#             "Breastfed exclusively until the end of Month Seven."
#         ),
#         "expected": {
#             "pedigree": None,
#             "consanguinity": "unrelated parents",
#             "antenatal_history": "severe vomiting and dehydration at week 28",
#             "perinatal_history": "cord prolapse managed successfully",
#             "postnatal_complications": "baby needed phototherapy for jaundice",
#             "breastfed_upto": "month seven"
#         }
#     },
#     {
#         # Adversarial Scenario 2
#         "text": (
#             "Lineage: no genetic red flags. Antenatal history: hypertension detected t1; perinatal history unremarkable. "
#             "After birth, no complications seen; breastfed for 2 mos."
#         ),
#         "expected": {
#             "pedigree": None,
#             "consanguinity": None,
#             "antenatal_history": "hypertension detected t1",
#             "perinatal_history": "unremarkable",
#             "postnatal_complications": "no complications seen",
#             "breastfed_upto": "2 mos"
#         }
#     }
# ]

# Setup medSpaCy pipeline if available
if SPACY_AVAILABLE:
    nlp = spacy.load("en_core_web_sm")
    nlp.add_pipe("sentencizer", first=True)
    try:
        nlp.add_pipe("medspacy_sectionizer", after="sentencizer")
        nlp.add_pipe("medspacy_context", last=True)
    except Exception:
        pass
    ruler = nlp.add_pipe("entity_ruler", before="ner")
    patterns = [
    # Pedigree / family history
    {"label":"PEDIGREE","pattern":"Pedigree"},
    {"label":"PEDIGREE","pattern":"Family history"},
    # remove generic "Lineage" to avoid overcapture
    # Consanguinity
    {"label":"CONSANGUINITY","pattern":"consanguinity"},
    {"label":"CONSANGUINITY","pattern":"consanguineous"},
    {"label":"CONSANGUINITY","pattern":"first-cousin"},
    {"label":"CONSANGUINITY","pattern":"unrelated parents"},  # new pattern
    # Antenatal events
    {"label":"ANTENATAL_EVENT","pattern":"preeclampsia"},
    {"label":"ANTENATAL_EVENT","pattern":"gestational diabetes"},
    {"label":"ANTENATAL_EVENT","pattern":"urinary tract infection"},
    {"label":"ANTENATAL_EVENT","pattern":"oligohydramnios"},
    # Perinatal events
    {"label":"PERINATAL_EVENT","pattern":"meconium-stained liquor"},
    {"label":"PERINATAL_EVENT","pattern":"shoulder dystocia"},
    {"label":"PERINATAL_EVENT","pattern":"cord prolapse"},
    {"label":"PERINATAL_EVENT","pattern":"premature rupture of membranes"},
    # Postnatal complications
    {"label":"POSTNATAL_EVENT","pattern":"jaundice"},
    {"label":"POSTNATAL_EVENT","pattern":"tachypnea"},
    {"label":"POSTNATAL_EVENT","pattern":"sepsis"},
    {"label":"POSTNATAL_EVENT","pattern":"hypoglycemia"},
    # Breastfeed duration
    {"label":"BREASTFEED_DURATION","pattern":[{"LIKE_NUM":True},{"LOWER":{"IN":["day","week","month","mos"]}}]}
]
    # Patterns for each field
    patterns += [{"label":"PEDIGREE","pattern":pt} for pt in ["Pedigree","Family history","Lineage"]]
    patterns += [{"label":"CONSANGUINITY","pattern":pt} for pt in ["consanguinity","consanguineous","first-cousin"]]
    patterns += [{"label":"ANTENATAL_EVENT","pattern":pt} for pt in ["preeclampsia","gestational diabetes","urinary tract infection","oligohydramnios"]]
    patterns += [{"label":"PERINATAL_EVENT","pattern":pt} for pt in ["meconium-stained liquor","shoulder dystocia","cord prolapse","premature rupture of membranes"]]
    patterns += [{"label":"POSTNATAL_EVENT","pattern":pt} for pt in ["jaundice","tachypnea","sepsis","hypoglycemia"]]
    patterns.append({
        "label":"BREASTFEED_DURATION",
        "pattern":[{"LIKE_NUM":True},{"LOWER":{"IN":["day","week","month","mos"]}}]
    })
    ruler.add_patterns(patterns)

# Regex extraction functions

def extract_pedigree_regex(text: str) -> str:
    m = re.search(r'(?:pedigree[:\s]*no|family history is negative for)\s*([\w\s-]+?)(?:;|\.|$)', text, re.IGNORECASE)
    if m:
        clause = m.group(1).strip().lower()
        return clause if 'negative for' in m.group(0).lower() else f"no {clause}"
    return None


def extract_consanguinity_regex(text: str) -> str:
    m = re.search(r'\b(no history of consanguinity|parents are unrelated|no consanguinity reported)\b', text, re.IGNORECASE)
    return m.group(1).lower() if m else None


def extract_antenatal_history_regex(text: str) -> str:
    m = re.search(r'antenatal (?:history|period)(?: was|:)?\s*(.*?)(?:\.|$)', text, re.IGNORECASE)
    return m.group(1).strip().lower() if m else None


def extract_perinatal_history_regex(text: str) -> str:
    m = re.search(r'perinatal (?:history:|events )?\s*(.*?)(?:\.|$)', text, re.IGNORECASE)
    return m.group(1).strip().lower() if m else None


def extract_postnatal_complications_regex(text: str) -> str:
    m = re.search(r'\bno postnatal complications\b', text, re.IGNORECASE)
    if m:
        return m.group(0).lower()
    m = re.search(r'postnatal (?:complications[:]?|)\s*(?:occurred\s*)?(.*?)(?:\.|$)', text, re.IGNORECASE)
    return m.group(1).strip().lower() if m else None


def extract_breastfed_upto_regex(text: str) -> str:
    m = re.search(r'breastfed (?:the infant )?(?:up to|for)\s*([\w\s]+?)(?:\.|$)', text, re.IGNORECASE)
    return m.group(1).strip().lower() if m else None

# Combine regex and medSpaCy extraction

def extract_all_fields(text: str) -> dict:
    """
    Extracts birth-history fields using regex and medSpaCy fallback.
    """
    results = {
        'pedigree': extract_pedigree_regex(text),
        'consanguinity': extract_consanguinity_regex(text),
        'antenatal_history': extract_antenatal_history_regex(text),
        'perinatal_history': extract_perinatal_history_regex(text),
        'postnatal_complications': extract_postnatal_complications_regex(text),
        'breastfed_upto': extract_breastfed_upto_regex(text),
    }
    if SPACY_AVAILABLE:
        doc = nlp(text)
        for ent in doc.ents:
            label = ent.label_
            # determine clause text from full sentence minus header
            clause = None
            if ':' in ent.sent.text:
                clause = ent.sent.text.split(':',1)[1].strip().lower()
            else:
                clause = ent.sent.text.strip().lower()
            # fill missing fields via medSpaCy
            if label == 'PEDIGREE' and not results['pedigree']:
                results['pedigree'] = clause
            elif label == 'CONSANGUINITY' and not results['consanguinity']:
                results['consanguinity'] = span = ent.text.strip().lower()
            elif label == 'ANTENATAL_EVENT' and not results['antenatal_history']:
                results['antenatal_history'] = ent.text.strip().lower()
            elif label == 'PERINATAL_EVENT' and not results['perinatal_history']:
                results['perinatal_history'] = clause
            elif label == 'POSTNATAL_EVENT' and not getattr(ent._, 'is_negated', False) and not results['postnatal_complications']:
                results['postnatal_complications'] = clause
            elif label == 'BREASTFEED_DURATION' and not results['breastfed_upto']:
                results['breastfed_upto'] = ent.text.strip().lower()
    return results

# Run tests and report accuracy
if __name__ == '__main__':
    total_fields = correct_fields = 0
    for idx, case in enumerate(TEST_CASES, 1):
        extracted = extract_all_fields(case['text'])
        print(f"\n--- Test Case {idx} ---")
        for field, expected in case['expected'].items():
            actual = extracted.get(field)
            if expected is None:
                match = actual is None
            elif actual is None:
                match = False
            else:
                match = isinstance(actual, str) and (expected in actual or actual in expected)
            status = '✓' if match else '✗'
            print(f"{field:25}: expected={expected!r}, actual={actual!r} {status}")
            total_fields += 1
            if match:
                correct_fields += 1
    accuracy = (correct_fields / total_fields * 100) if total_fields else 0
    print(f"\nOverall accuracy: {accuracy:.2f}% ({correct_fields}/{total_fields})")



--- Test Case 1 ---
pedigree                 : expected='no familial genetic disorders', actual=None ✗
consanguinity            : expected='no history of consanguinity', actual='no history of consanguinity' ✓
antenatal_history        : expected='mild urinary tract infection treated at 32 weeks', actual='remarkable only for a mild urinary tract infection treated at 32 weeks' ✓
perinatal_history        : expected='unremarkable', actual='were unremarkable' ✓
postnatal_complications  : expected='no postnatal complications', actual='no postnatal complications' ✓
breastfed_upto           : expected='eight weeks', actual='eight weeks' ✓

--- Test Case 2 ---
pedigree                 : expected='negative for neuromuscular diseases', actual='neuromuscular diseases' ✓
consanguinity            : expected='parents are unrelated', actual='parents are unrelated' ✓
antenatal_history        : expected='gestational diabetes managed with diet', actual='included gestational diabetes managed with diet' ✓


In [50]:
# Test suite to validate Birth History text extraction using medSpaCy and sentence-based clause splitting
import spacy
from test_birth_history_examples import TEST_CASES

# Setup spaCy + medSpaCy pipeline (medSpaCy_sectionizer/context not strictly required here)
nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("sentencizer", first=True)

# Keywords for each field
FIELD_KEYWORDS = {
    'pedigree': ['pedigree', 'family history', 'lineage'],
    'consanguinity': ['unrelated parents', 'consanguinity', 'consanguineous'],
    'antenatal_history': ['antenatal history', 'antenatal period'],
    'perinatal_history': ['perinatal history', 'perinatal events', 'perinatal'],
    'postnatal_complications': ['postnatal complications', 'postnatal events', 'postnatal'],
    'breastfed_upto': ['breastfed up to', 'breastfed for', 'breastfed until', 'breastfed'],
}

def clean_clause(raw: str) -> str:
    """
    Trim leading/trailing whitespace and punctuation, lower-case.
    """
    return raw.strip(" .;:-").lower()


def extract_clause(s: str, phrase: str) -> str:
    """
    Extract substring after the given phrase up to common delimiters.
    """
    sl = s.lower()
    start = sl.find(phrase)
    if start < 0:
        return None
    # Move past the phrase
    raw = s[start + len(phrase):]
    # Stop at first delimiter
    for delim in [';', '.', ':']:
        if delim in raw:
            raw = raw.split(delim, 1)[0]
    return clean_clause(raw)


def extract_all_fields(text: str) -> dict:
    """
    Extract birth-history fields by sentence segmentation and keyword-based splitting.
    """
    doc = nlp(text)
    results = {field: None for field in FIELD_KEYWORDS}

    for sent in doc.sents:
        s = sent.text.strip()
        sl = s.lower()
        for field, phrases in FIELD_KEYWORDS.items():
            if results[field] is not None:
                continue
            for phrase in phrases:
                if phrase in sl:
                    clause = extract_clause(s, phrase)
                    if clause:
                        # Post-process breastfed clause to isolate duration
                        if field == 'breastfed_upto':
                            if 'up to' in clause:
                                clause = clause.split('up to', 1)[1].strip()
                            elif 'for' in clause:
                                clause = clause.split('for', 1)[1].strip()
                        results[field] = clause
                        break
            # break outer loop if field just filled
            if results[field] is not None:
                break
    return results

# Run tests and report accuracy
if __name__ == '__main__':
    total_fields = correct_fields = 0
    for idx, case in enumerate(TEST_CASES, 1):
        extracted = extract_all_fields(case['text'])
        print(f"\n--- Test Case {idx} ---")
        for field, expected in case['expected'].items():
            actual = extracted.get(field)
            match = (actual is None and expected is None) or (actual and expected and (expected in actual or actual in expected))
            status = '✓' if match else '✗'
            print(f"{field:25}: expected={expected!r}, actual={actual!r} {status}")
            total_fields += 1
            if match:
                correct_fields += 1
    accuracy = (correct_fields / total_fields * 100) if total_fields else 0
    print(f"\nOverall accuracy: {accuracy:.2f}% ({correct_fields}/{total_fields})")



--- Test Case 1 ---
pedigree                 : expected='no familial genetic disorders', actual='with no familial genetic disorders and no history of consanguinity' ✓
consanguinity            : expected='no history of consanguinity', actual=None ✗
antenatal_history        : expected='mild urinary tract infection treated at 32 weeks', actual='was remarkable only for a mild urinary tract infection treated at 32 weeks' ✓
perinatal_history        : expected='unremarkable', actual='were unremarkable' ✓
postnatal_complications  : expected='no postnatal complications', actual=', and the mother breastfed the infant up to eight weeks' ✗
breastfed_upto           : expected='eight weeks', actual=None ✗

--- Test Case 2 ---
pedigree                 : expected='negative for neuromuscular diseases', actual='is negative for neuromuscular diseases' ✓
consanguinity            : expected='parents are unrelated', actual=None ✗
antenatal_history        : expected='gestational diabetes managed with diet',

In [53]:
import re

# Attempt to import word2number for spelled number conversion
try:
    from word2number import w2n
    W2N_AVAILABLE = True
except ImportError:
    w2n = None
    W2N_AVAILABLE = False

def extract_birth_weight(text: str) -> float:
    """
    Extracts birth weight in kilograms from a clinical text.
    Returns the weight in kg, or None if no weight is found.
    """
    # Numeric pattern for kg or g
    numeric_match = re.search(r'(\d+(?:[\d,]*)(?:\.\d+)?)\s*(kg|kilograms|g|grams|kilos)', text, re.IGNORECASE)
    if numeric_match:
        value = float(numeric_match.group(1).replace(',', ''))
        unit = numeric_match.group(2).lower()
        if unit in ['g', 'grams']:
            value = value / 1000
        return value

    # Spelled-out number pattern for kg/kilos and g/grams
    if W2N_AVAILABLE:
        spelled_match = re.search(
                r'([a-z\s\-]+?)\s*(kilograms|kilogram|kilos|kilo|grams|gram|kg|g)\b',
                text,
                re.IGNORECASE
            )

        if spelled_match:
            words = spelled_match.group(1)
            unit = spelled_match.group(2).lower()
            try:
                num = w2n.word_to_num(words)
                value = float(num)
                if unit in ['g', 'grams']:
                    value = value / 1000
                return value
            except ValueError:
                pass

    return None

# Test the function on various inputs
test_sentences = [
    "At birth, the infant weighed 2.5 kg and cried immediately.",
    "He did not cry immediately, and weighed two thousand five hundred grams at birth.",
    "The baby weighed two point five kilograms at birth."
]

for sentence in test_sentences:
    weight_kg = extract_birth_weight(sentence)
    print(f"Sentence: {sentence}\nExtracted weight (kg): {weight_kg}\n")


Sentence: At birth, the infant weighed 2.5 kg and cried immediately.
Extracted weight (kg): 2.5

Sentence: He did not cry immediately, and weighed two thousand five hundred grams at birth.
Extracted weight (kg): 2.5

Sentence: The baby weighed two point five kilograms at birth.
Extracted weight (kg): 2.5



In [ ]:
import time
import re
from faster_whisper import WhisperModel
import spacy
from spacy.matcher import PhraseMatcher
from radio_extractor.config import KEYWORD_MAP
from word2number import w2n  # Assuming you have word2number for spelled-out numbers
from dataset import Dataset


# Initialize Faster Whisper for Speech-to-Text
def transcribe_audio(audio_file: str):
    model_size = "small"
    whisper_model = WhisperModel(model_size, device="cpu", compute_type="int8")

    stt_start = time.time()
    segments, info = whisper_model.transcribe(audio_file, beam_size=5)
    stt_end = time.time()

    print(f"Transcription took {stt_end - stt_start:.2f} seconds")
    return segments

# Initialize spaCy and set up PhraseMatcher
nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("sentencizer", first=True)
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

# Build PhraseMatcher for all radio fields
for field, synonyms in KEYWORD_MAP.items():
    patterns = [nlp.make_doc(expr) for expr in synonyms.keys()]
    matcher.add(field, patterns)

# Context keywords for disambiguation of 'assisted'
CONTEXT_KEYWORDS = {
    "conception_mode": ["conceived", "fertilization", "art"],
    "delivery_mode": ["deliver", "delivery", "section", "vaginal"],
}

# Regex fallback for radio fields
REGEX_PATTERNS = {}
for field, synonyms in KEYWORD_MAP.items():
    keys = sorted(synonyms.keys(), key=lambda x: -len(x))
    pattern = r"\b(?:" + r"|".join(re.escape(k) for k in keys) + r")\b"
    REGEX_PATTERNS[field] = re.compile(pattern, re.IGNORECASE)

# Extract radio fields using spaCy PhraseMatcher and regex fallback
def extract_radio_fields(text: str) -> dict:
    results = {}
    doc = nlp(text)
    matches = matcher(doc)

    # 1. spaCy-based extraction
    for match_id, start, end in matches:
        field = doc.vocab.strings[match_id]
        span = doc[start:end]
        span_text = span.text.lower().strip()
        sent = span.sent
        sent_text = sent.text.lower()
        
        # Handle negation for 'cried_at_birth'
        if field == "cried_at_birth":
            neg = bool(re.search(r"\b(did not cry|didn't cry|failed to cry|no cry)\b", sent_text))
            results[field] = "No" if neg else "Yes"
            continue
        
        # Map only if synonym in map
        option = KEYWORD_MAP[field].get(span_text)
        if option:
            # Disambiguate 'assisted'
            if span_text == "assisted" and not any(ctx in sent_text for ctx in CONTEXT_KEYWORDS.get(field, [])):
                continue
            results[field] = option

    # 2. Regex fallback for missing fields
    for field, pattern in REGEX_PATTERNS.items():
        if field not in results:
            m = pattern.search(text)
            if m:
                key = m.group(0).lower()
                results[field] = KEYWORD_MAP[field].get(key)
    return results

# Extract birth weight using regex
def extract_birth_weight(text: str) -> float:
    numeric_match = re.search(
        r'(\d+(?:[\d,]*)(?:\.\d+)?)\s*(kg|kilograms|g|grams|kilos)',
        text,
        re.IGNORECASE
    )
    if numeric_match:
        value = float(numeric_match.group(1).replace(',', ''))
        unit = numeric_match.group(2).lower()
        if unit in ['g', 'grams']:
            value = value / 1000
        return value

    if w2n:
        spelled_match = re.search(
            r'([a-z\s\-\d]+?)\s*(kg|kilograms|kilos)',
            text,
            re.IGNORECASE
        )
        if spelled_match:
            words = spelled_match.group(1)
            try:
                num = w2n.word_to_num(words)
                return float(num)
            except Exception:
                pass
    return None

def extract_pedigree_regex(text: str) -> str:
    m = re.search(r'(?:pedigree[:\s]*no|family history is negative for|pedigree was|pedigree|family history|lineage| genetic)\s*([\w\s-]+?)(?:;|\.|$)', text, re.IGNORECASE)
    if m:
        clause = m.group(1).strip().lower()
        return clause if 'negative for' in m.group(0).lower() or 'unremarkable' in m.group(0).lower() else f"no {clause}"
    return None


def extract_consanguinity_regex(text: str) -> str:
    m = re.search(r'\b(no history of consanguinity|parents are unrelated|no consanguinity reported|cousin marriage)\b', text, re.IGNORECASE)
    return m.group(1).lower() if m else None


def extract_antenatal_history_regex(text: str) -> str:
    m = re.search(r'antenatal (?:history|period)(?: was|:)?\s*(.*?)(?:\.|$)', text, re.IGNORECASE)
    return m.group(1).strip().lower() if m else None


def extract_perinatal_history_regex(text: str) -> str:
    m = re.search(r'perinatal (?:history:|events )?\s*(.*?)(?:\.|$)', text, re.IGNORECASE)
    return m.group(1).strip().lower() if m else None


def extract_postnatal_complications_regex(text: str) -> str:
    m = re.search(r'\bno postnatal complications\b', text, re.IGNORECASE)
    if m:
        return m.group(0).lower()
    m = re.search(r'postnatal (?:complications[:]?|)\s*(?:occurred\s*)?(.*?)(?:\.|$)', text, re.IGNORECASE)
    return m.group(1).strip().lower() if m else None


def extract_breastfed_upto_regex(text: str) -> str:
    m = re.search(r'\b(?:breastfed|breastfeeding)\s*(?:the infant )?(?:exclusively )?(?:up to|for|continued for|was continued)\s*([\w\s]+?)(?:months?|days?|years?|mos|yrs|\.|$)', text, re.IGNORECASE)
    return m.group(1).strip().lower() + " months" if m else None

# Extract other fields using regex
def extract_other_fields(text: str) -> dict:
    return {
        "pedigree": extract_pedigree_regex(text),
        "consanguinity": extract_consanguinity_regex(text),
        "antenatal_history": extract_antenatal_history_regex(text),
        "perinatal_history": extract_perinatal_history_regex(text),
        "postnatal_complications": extract_postnatal_complications_regex(text),
        "breastfed_upto": extract_breastfed_upto_regex(text),
    }

# Function to combine all extraction methods into one pipeline
def extract_birth_history_from_audio(audio_file: str):
    # Step 1: Transcribe audio
    segments = transcribe_audio(audio_file)
    
    # Step 2: Combine all text segments into a single string
    full_text = " ".join([segment.text for segment in segments])
    print(f"Transcribed text: {full_text}")
    print("---------------------------------------------------------------------------------------------------------")
    print()

    # Step 3: Extract relevant fields using all extraction methods
    birth_history = {}
    birth_history.update(extract_radio_fields(full_text))
    birth_history["birth_weight"] = extract_birth_weight(full_text)
    birth_history.update(extract_other_fields(full_text))

    return birth_history

def extract_birth_history_from_dataset(text: str):
    """
    Extracts birth history fields from a given text string.
    """    
    birth_history = {}
    birth_history.update(extract_radio_fields(text))
    birth_history["birth_weight"] = extract_birth_weight(text)
    birth_history.update(extract_other_fields(text))
    return birth_history

# Example usage
audio_file = "Recording (2).m4a"
birth_history = extract_birth_history_from_audio(audio_file)

print(birth_history)
# for entry in Dataset:
#     full_text = entry["diagnosis_text"]
#     # extract_birth_history_from_dataset(full_text)
#     birth_history = extract_birth_history_from_dataset(full_text)
    
#     print(entry['diagnosis_text'])
#     print()
#     print(birth_history)

    
    
#     print("---------------------------------------------------------------------------------------------------------")
       


Transcription took 2.12 seconds
Transcribed text:  The family history shows no significant pedigree, known continuity reported between the parents.  The original history was normal with no complications.  Perinatal events included a slight delay in crying after the birth.  The baby was conceived naturally and delivered by normal vaginal delivery at term.  Birth weight was recorded as 3.2 kg.  There were no postal complications.  Infant was breastfed exclusively for 6 months.
---------------------------------------------------------------------------------------------------------

{'delivery_mode': 'NVD', 'conception_mode': 'Natural', 'term': 'Term', 'birth_weight': 3.2, 'pedigree': None, 'consanguinity': None, 'antenatal_history': None, 'perinatal_history': 'included a slight delay in crying after the birth', 'postnatal_complications': None, 'breastfed_upto': '6 months'}


In [17]:
from dataset import Dataset

def extract_birth_history_from_text(full_text: str):
    # Step 1: Transcribe audio
    # segments = transcribe_audio(audio_file)
    
    # Step 2: Combine all text segments into a single string
    # full_text = " ".join([segment.text for segment in segments])
    # print(f"Transcribed text: {full_text}")
    # print("---------------------------------------------------------------------------------------------------------")
    # print()

    # Step 3: Extract relevant fields using all extraction methods
    birth_history = {}
    birth_history.update(extract_radio_fields(full_text))
    birth_history["birth_weight"] = extract_birth_weight(full_text)
    birth_history.update(extract_other_fields(full_text))
    

    
for entry in Dataset:
    full_text = entry["diagnosis_text"]
    # print(type(full_text))
    # print(full_text)
    birth_history = extract_birth_history_from_text(full_text)
    
    print(birth_history)
    


None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
